# Webull Thailand Endpoint Tutorial: Authentication Token

## Visual Quick Start

ดูรูปภาพรวมก่อน แล้วค่อย run cell ด้านล่างทีละช่อง.

![3 ขั้นตอนเริ่มต้นใช้ Webull Open API](../docs/assets/webull-openapi-quickstart/01-cover.png)

![วิธีขอ App Key และ App Secret](../docs/assets/webull-openapi-quickstart/03-app-key-secret.png)

![วิธีขอ Access Token](../docs/assets/webull-openapi-quickstart/05-access-token.png)

Learning goals:
- รู้ว่า endpoint ในหมวดนี้ใช้ทำอะไร
- เริ่มจาก offline sample response ก่อนใช้ credential จริง
- บันทึก raw JSON ทุก endpoint เพื่อกลับมาตรวจ response ได้
- แปลง response เป็น DataFrame หรือกราฟเมื่อเหมาะสม

Official sources:
- https://developer.webull.co.th/apis/docs/reference/custom/authentication/
- https://developer.webull.co.th/apis/docs/reference/trade-api/create-token/
- https://developer.webull.co.th/apis/docs/reference/trade-api/check-token/

Endpoints covered:
- `POST /openapi/auth/token/create` - Create Token
- `POST /openapi/auth/token/check` - Check Token


## Safety Defaults

ค่า default คือ offline mode (`WEBULL_TUTORIAL_LIVE=0`) จึง run ได้โดยไม่ต้องมี credential.
ถ้าจะยิง API จริง ให้ตั้งค่า `.env.webull-th` หรือ `.env` แล้วเปิด `WEBULL_TUTORIAL_LIVE=1`.

### โค้ดช่องถัดไปทำอะไร

- กำหนดชื่อ notebook เพื่อใช้ตั้งชื่อโฟลเดอร์ output
- สร้างรายการ endpoint ที่ notebook นี้จะเรียก เช่น method, path, parameter และ body
- ยังไม่ยิง API จริง เป็นแค่การเตรียมแผนที่ endpoint ให้ cell ถัด ๆ ไปใช้ร่วมกัน


In [ ]:
NOTEBOOK_SLUG = "00-auth-token"
ENDPOINTS = [{'name': 'create_token',
  'label': 'Create Token',
  'method': 'POST',
  'path': '/openapi/auth/token/create',
  'purpose': 'Create access token; token may require app verification before '
             'normal use.',
  'params': {},
  'body': {'app_key': 'from_env'}},
 {'name': 'check_token',
  'label': 'Check Token',
  'method': 'POST',
  'path': '/openapi/auth/token/check',
  'purpose': 'Check whether a token is NORMAL, PENDING, INVALID, or EXPIRED.',
  'params': {},
  'body': {'access_token': 'from_env_or_sample'}}]


### โค้ดช่องถัดไปทำอะไร

- import library ที่ใช้กับ API, JSON, DataFrame และไฟล์ output
- ตั้งค่า `BASE_URL`, host ไทย, live/offline mode และโฟลเดอร์ output
- อ่านค่า credential จาก `.env.webull-th` หรือ `.env` แต่ไม่พิมพ์ secret ออกมาตรง ๆ
- สร้าง helper สำหรับ redact ข้อมูลส่วนตัว, save JSON และ export CSV


In [ ]:
import base64
import hashlib
import hmac
import json
import os
import uuid
from datetime import UTC, datetime
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests

try:
    NOTEBOOK_DISPLAY = display
except NameError:
    def display(value: object) -> None:
        print(value)
else:
    display = NOTEBOOK_DISPLAY

BASE_URL = "https://api.webull.co.th"
HOST = "api.webull.co.th"
REGION = "th"
LIVE_MODE = os.getenv("WEBULL_TUTORIAL_LIVE", "0") == "1"
OUTPUT_ROOT = Path(os.getenv("WEBULL_TUTORIAL_OUTPUT_DIR", "outputs/webull-th-endpoints"))
OUTPUT_DIR = OUTPUT_ROOT / NOTEBOOK_SLUG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_env_file(path: str | Path) -> dict[str, str]:
    env_path = Path(path)
    if not env_path.exists():
        return {}
    values: dict[str, str] = {}
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        key, value = stripped.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def env_value(key: str, file_values: dict[str, str], default: str = "") -> str:
    return os.getenv(key, file_values.get(key, default)).strip()


def redact(value: object, keep: int = 4) -> str:
    text = str(value or "")
    if not text:
        return "<missing>"
    if len(text) <= keep * 2:
        return "*" * len(text)
    return f"{text[:keep]}...{text[-keep:]}"


def redact_nested(value: object) -> object:
    sensitive = {"account_id", "accountId", "account_number", "accountNumber", "token"}
    if isinstance(value, dict):
        return {
            key: redact(val, keep=3) if key in sensitive else redact_nested(val)
            for key, val in value.items()
        }
    if isinstance(value, list):
        return [redact_nested(item) for item in value]
    return value


def save_json(name: str, payload: object) -> Path:
    path = OUTPUT_DIR / f"{name}.json"
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path


def save_records_csv(name: str, payload: object) -> pd.DataFrame:
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = payload.get("data", payload)
    else:
        records = []
    if isinstance(records, dict):
        records = [records]
    frame = pd.DataFrame(records)
    if not frame.empty:
        frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    return frame


env_file = Path(".env.webull-th") if Path(".env.webull-th").exists() else Path(".env")
file_values = read_env_file(env_file)

APP_KEY = env_value("WEBULL_APP_KEY", file_values)
APP_SECRET = env_value("WEBULL_APP_SECRET", file_values)
ACCESS_TOKEN = env_value("WEBULL_ACCESS_TOKEN", file_values)
ACCOUNT_ID = env_value("WEBULL_ACCOUNT_ID", file_values)

if LIVE_MODE and (not APP_KEY or not APP_SECRET):
    raise RuntimeError("Live mode requires WEBULL_APP_KEY and WEBULL_APP_SECRET")

print(
    {
        "live_mode": LIVE_MODE,
        "base_url": BASE_URL,
        "env_file": str(env_file),
        "app_key": redact(APP_KEY),
        "secret_present": bool(APP_SECRET),
        "token_present": bool(ACCESS_TOKEN),
        "account_id": redact(ACCOUNT_ID),
        "output_dir": str(OUTPUT_DIR),
    }
)


## Endpoint Map

หมวดนี้โฟกัส: Create Token และ Check Token สำหรับ server-to-server authentication

### โค้ดช่องถัดไปทำอะไร

- แสดง endpoint ทั้งหมดใน notebook นี้เป็นตารางอ่านง่าย
- ใช้ตรวจเร็ว ๆ ว่าแต่ละ endpoint เป็น `GET` หรือ `POST`
- ช่วยให้เห็น path จริงก่อนเริ่มเรียก API หรืออ่าน sample response


In [ ]:
pd.DataFrame(
    [
        {
            "name": endpoint["name"],
            "method": endpoint["method"],
            "path": endpoint["path"],
            "purpose": endpoint["purpose"],
        }
        for endpoint in ENDPOINTS
    ]
)


## Offline Samples

Cell ถัดไปคือ sample response สำหรับโหมด offline. โครงสร้างนี้ใช้เพื่อสอนการอ่าน JSON,
ไม่ใช่ข้อมูลตลาดแบบ real-time.

### โค้ดช่องถัดไปทำอะไร

- สร้างตัวแปร `SAMPLE_RESPONSES` เป็นข้อมูลจำลองสำหรับทุก endpoint
- ทำให้ run notebook ได้ทันที แม้ยังไม่มี App Key, App Secret หรือ token
- ใช้ฝึกอ่านโครงสร้าง JSON ก่อนค่อยเปิด live mode


In [ ]:
SAMPLE_RESPONSES = {'create_token': {'access_token': 'offline-demo-token',
                  'expires_in_days': 15,
                  'status': 'PENDING',
                  'message': 'Verify the token in the Webull app before live '
                             'use.'},
 'check_token': {'access_token': 'offline-demo-token',
                 'status': 'NORMAL',
                 'checked_at': '2026-07-06T00:00:00Z'}}


## Request Helper

Helper นี้สร้าง signed request สำหรับ live mode ด้วย `HMAC-SHA256`.
ใน offline mode จะคืน sample response ทันที.

### โค้ดช่องถัดไปทำอะไร

- สร้าง header ที่ Webull ต้องใช้ เช่น `x-app-key`, timestamp, nonce และ signature
- รวม query/body ให้เป็น request ที่ส่งด้วย `requests.request`
- ถ้าอยู่ offline mode จะไม่ยิง internet และคืน sample response แทน
- ถ้า live API error จะ raise error พร้อม status code และ payload เพื่อ debug


In [ ]:
def build_signature_headers(
    *,
    path: str,
    query_params: dict[str, object],
    body: dict[str, object] | None = None,
    algorithm: str = "HMAC-SHA256",
) -> dict[str, str]:
    timestamp = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
    nonce = str(uuid.uuid4())
    body_str = json.dumps(body, separators=(",", ":"), ensure_ascii=False) if body else ""

    sign_params = {key: str(value) for key, value in query_params.items() if value is not None}
    sign_params.update(
        {
            "host": HOST,
            "x-app-key": APP_KEY,
            "x-signature-algorithm": algorithm,
            "x-signature-nonce": nonce,
            "x-signature-version": "1.0",
            "x-timestamp": timestamp,
        }
    )

    sorted_params = "&".join(f"{key}={value}" for key, value in sorted(sign_params.items()))
    string_to_sign = f"{path}&{sorted_params}"
    if body_str:
        body_md5 = hashlib.md5(body_str.encode()).hexdigest().upper()
        string_to_sign = f"{string_to_sign}&{body_md5}"

    encoded_string = quote(string_to_sign, safe="")
    digest = hashlib.sha256 if algorithm == "HMAC-SHA256" else hashlib.sha1
    signature = base64.b64encode(
        hmac.new((APP_SECRET + "&").encode(), encoded_string.encode(), digest).digest()
    ).decode()

    headers = {
        "Accept": "application/json",
        "Accept-Encoding": "gzip",
        "Content-Type": "application/json",
        "x-version": "v2",
        "x-app-key": APP_KEY,
        "x-timestamp": timestamp,
        "x-signature-version": "1.0",
        "x-signature-algorithm": algorithm,
        "x-signature-nonce": nonce,
        "x-signature": signature,
        "x-webull-client-source": "sdk",
    }
    if ACCESS_TOKEN:
        headers["x-access-token"] = ACCESS_TOKEN
    return headers


def call_endpoint(
    endpoint: dict[str, object],
    *,
    sample: object,
    query_params: dict[str, object] | None = None,
    body: dict[str, object] | None = None,
) -> object:
    if not LIVE_MODE:
        return sample

    query = {key: value for key, value in (query_params or {}).items() if value is not None}
    request_body = body or None
    path = str(endpoint["path"])
    headers = build_signature_headers(path=path, query_params=query, body=request_body)
    method = str(endpoint["method"]).upper()
    response = requests.request(
        method,
        f"{BASE_URL}{path}",
        headers=headers,
        params=query,
        json=request_body,
        timeout=30,
    )
    try:
        payload = response.json()
    except ValueError:
        payload = {"raw_text": response.text}
    if response.status_code >= 400:
        raise RuntimeError({"status_code": response.status_code, "payload": payload})
    return payload


print("request helpers ready")


## Run Endpoints

ทุก response จะถูก redacted แล้ว save เป็น JSON ใน `OUTPUT_DIR`.

### โค้ดช่องถัดไปทำอะไร

- วนเรียก endpoint ทุกตัวใน `ENDPOINTS`
- แทน placeholder เช่น account id ด้วยค่าจาก env หรือ sample
- redact token/account id ก่อนเก็บใน `results`
- save raw JSON แยกเป็นไฟล์เพื่อให้เปิดดูย้อนหลังหรือแคปหน้าจอได้


In [ ]:
def resolve_endpoint_value(value: object) -> object:
    if value == "from_env_or_sample":
        return ACCOUNT_ID or "offline-account-001"
    if value == "from_env":
        return APP_KEY or "offline-app-key"
    if isinstance(value, dict):
        return {key: resolve_endpoint_value(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_endpoint_value(item) for item in value]
    return value


results = {}
saved_paths = {}

for endpoint in ENDPOINTS:
    name = endpoint["name"]
    query_params = resolve_endpoint_value(endpoint.get("params") or {})
    body = resolve_endpoint_value(endpoint.get("body"))
    payload = call_endpoint(
        endpoint,
        sample=SAMPLE_RESPONSES[name],
        query_params=query_params,
        body=body,
    )
    clean_payload = redact_nested(payload)
    results[name] = clean_payload
    saved_paths[name] = str(save_json(name, clean_payload))

print({"endpoints": list(results), "saved_paths": saved_paths})


## Inspect and Export

### โค้ดช่องถัดไปทำอะไร

- นำผลลัพธ์จาก `results` มาแปลงเป็นตาราง, CSV หรือกราฟตามชนิด endpoint
- ช่วยให้คนทั่วไปอ่าน response ได้ง่ายกว่า raw JSON
- output สำคัญจะถูกเก็บไว้ใน `OUTPUT_DIR` เพื่อกลับมาเปิดดูได้ภายหลัง


In [ ]:
summary = pd.DataFrame(
    [
        {
            "endpoint": name,
            "status": payload.get("status") if isinstance(payload, dict) else None,
            "saved_json": saved_paths[name],
        }
        for name, payload in results.items()
    ]
)
summary.to_csv(OUTPUT_DIR / "auth-token-summary.csv", index=False)
summary


## Guardrails and Non-Goals

            - อย่าใส่ App Secret ลง notebook, screenshot, หรือ git history
- Token จริงเป็น credential; output ใน notebook นี้ต้อง redact ก่อนแชร์


## Exercises

            1. ลองปิด live mode แล้วดูว่า sample response ถูก save ที่ไหน
2. เปิดไฟล์ JSON ที่ save แล้วดู field `status` ของ token
3. เขียน checklist สั้น ๆ ว่าเมื่อ token เป็น PENDING ต้องทำอะไรต่อ
